# Group-to-Group Recommendation System
Vector-based similarity search across 200 groups using commodity overlap,
target-role overlap, and geographic proximity.
No user data. No external connections. Groups only.


In [4]:
# !pip install openpyxl
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Groups dataset (xlsx saved with .csv extension)
df = pd.read_excel('groups_targeting_india.csv', engine='openpyxl')

# Parse comma-separated fields into Python lists
df['commodity_list']   = df['commodities'].apply(
    lambda x: sorted(set(c.strip() for c in str(x).split(',')))
)
df['target_role_list'] = df['target_roles'].apply(
    lambda x: sorted(set(r.strip() for r in str(x).split(',')))
)

print(f"Dataset : {df.shape[0]} groups, {df.shape[1]} columns")
print()
df.head(8)


Dataset : 200 groups, 7 columns



,group_id,commodities,target_roles,latitude,longitude,commodity_list,target_role_list
0,1,rice,trader,21.1458,79.0882,[rice],[trader]
1,2,rice,trader,21.1458,79.0882,[rice],[trader]
2,3,rice,trader,21.1458,79.0882,[rice],[trader]
3,4,rice,trader,21.1458,79.0882,[rice],[trader]
4,5,rice,trader,21.1458,79.0882,[rice],[trader]
5,6,rice,trader,21.1458,79.0882,[rice],[trader]
6,7,rice,trader,21.1458,79.0882,[rice],[trader]
7,8,rice,trader,21.1458,79.0882,[rice],[trader]


## Parameters — tune everything here

In [5]:
# ══════════════════════════════════════════════════════════════════════════
#  PARAMETERS — one place, nothing hardcoded below
# ══════════════════════════════════════════════════════════════════════════

COMMODITY_BOOST = 1.0   # raise to make commodity overlap matter more
ROLE_BOOST      = 1.5   # raise to make target-role overlap matter more
GEO_BOOST       = 3.0   # raise to make proximity matter more
                         # lower any boost toward 0 to de-emphasise that factor

# ── Vocabularies (auto-derived from data, not hardcoded) ──────────────────
ALL_COMMODITIES = sorted(set(
    c for lst in df['commodity_list'] for c in lst
))
ALL_ROLES = sorted(set(
    r for lst in df['target_role_list'] for r in lst
))

# ── Dimension names (for display / charts) ────────────────────────────────
COMM_DIMS = [f'has_{c}'      for c in ALL_COMMODITIES]   # 3 dims
ROLE_DIMS = [f'targets_{r}'  for r in ALL_ROLES]         # 3 dims
GEO_DIMS  = ['geo_x','geo_y','geo_z']                    # 3 dims
DIM_LABELS = COMM_DIMS + ROLE_DIMS + GEO_DIMS            # 9 dims total

n_comm = len(ALL_COMMODITIES)
n_role = len(ALL_ROLES)
n_geo  = 3
total  = n_comm + n_role + n_geo

print(f"ALL_COMMODITIES : {ALL_COMMODITIES}")
print(f"ALL_ROLES       : {ALL_ROLES}")
print()
print(f"Vector layout  ({total} dims total)")
print(f"  [0 : {n_comm}]   commodity   {COMM_DIMS}   boost={COMMODITY_BOOST}")
print(f"  [{n_comm} : {n_comm+n_role}]   role        {ROLE_DIMS}   boost={ROLE_BOOST}")
print(f"  [{n_comm+n_role} : {total}]   geo         {GEO_DIMS}   boost={GEO_BOOST}")


ALL_COMMODITIES : ['cotton', 'rice', 'sugar']
ALL_ROLES       : ['broker', 'exporter', 'trader']

Vector layout  (9 dims total)
  [0 : 3]   commodity   ['has_cotton', 'has_rice', 'has_sugar']   boost=1.0
  [3 : 6]   role        ['targets_broker', 'targets_exporter', 'targets_trader']   boost=1.5
  [6 : 9]   geo         ['geo_x', 'geo_y', 'geo_z']   boost=3.0


## Encoding functions

In [6]:
# ── Commodity ─────────────────────────────────────────────────────────────────
# Binary presence of each commodity.  Same vector stored AND queried (symmetric).
def encode_commodity(commodity_list):
    vec = np.zeros(n_comm)
    for c in commodity_list:
        if c in ALL_COMMODITIES:
            vec[ALL_COMMODITIES.index(c)] = 1.0
    return vec * COMMODITY_BOOST


# ── Target roles ──────────────────────────────────────────────────────────────
# Binary presence of each targeted role.
# Two groups both targeting 'broker' → high dot product on that dim.
def encode_roles(target_role_list):
    vec = np.zeros(n_role)
    for r in target_role_list:
        if r in ALL_ROLES:
            vec[ALL_ROLES.index(r)] = 1.0
    return vec * ROLE_BOOST


# ── Geo ───────────────────────────────────────────────────────────────────────
# Projects lat/lon onto the unit sphere (3-D Cartesian).
# Cosine similarity of two such vectors equals cos(central angle) —
# nearby locations produce a high dot product.
def encode_geo(lat, lon):
    lat_r = np.radians(lat)
    lon_r = np.radians(lon)
    x = np.cos(lat_r) * np.cos(lon_r)
    y = np.cos(lat_r) * np.sin(lon_r)
    z = np.sin(lat_r)
    return np.array([x, y, z]) * GEO_BOOST


# ── Full group vector (commodity + role + geo) ────────────────────────────────
def encode_group(commodity_list, target_role_list, lat, lon):
    return np.concatenate([
        encode_commodity(commodity_list),
        encode_roles(target_role_list),
        encode_geo(lat, lon),
    ])


# ── Quick sanity check ────────────────────────────────────────────────────────
print("COMM_DIMS  :", COMM_DIMS)
print("ROLE_DIMS  :", ROLE_DIMS)
print()

test_cases = [
    (['rice'],          ['trader']),
    (['cotton','rice'], ['broker','exporter']),
    (['sugar'],         ['broker','trader','exporter']),
]
for comms, roles in test_cases:
    v = encode_group(comms, roles, 20.0, 78.0)
    print(f"  comms={comms}  roles={roles}")
    print(f"    → comm  {v[:n_comm].round(3)}")
    print(f"    → role  {v[n_comm:n_comm+n_role].round(3)}")
    print()


COMM_DIMS  : ['has_cotton', 'has_rice', 'has_sugar']
ROLE_DIMS  : ['targets_broker', 'targets_exporter', 'targets_trader']

  comms=['rice']  roles=['trader']
    → comm  [0. 1. 0.]
    → role  [0.  0.  1.5]

  comms=['cotton', 'rice']  roles=['broker', 'exporter']
    → comm  [1. 1. 0.]
    → role  [1.5 1.5 0. ]

  comms=['sugar']  roles=['broker', 'trader', 'exporter']
    → comm  [0. 0. 1.]
    → role  [1.5 1.5 1.5]



## Build group vectors and vector DB

In [7]:
# Build one vector per group — used both as stored candidates and as queries
group_vectors = np.array([
    encode_group(
        row['commodity_list'],
        row['target_role_list'],
        row['latitude'],
        row['longitude'],
    )
    for _, row in df.iterrows()
])

df['vector'] = group_vectors.tolist()

# ── In-memory vector DB ───────────────────────────────────────────────────────
group_db = []
for idx, row in df.iterrows():
    group_db.append({
        'id':     int(row['group_id']),
        'vector': group_vectors[idx],
        'meta': {
            'group_id':        int(row['group_id']),
            'commodities':     row['commodities'],
            'commodity_list':  row['commodity_list'],
            'target_roles':    row['target_roles'],
            'target_role_list': row['target_role_list'],
            'latitude':        row['latitude'],
            'longitude':       row['longitude'],
        }
    })

print(f"Group vector DB : {len(group_db)} records,  {total} dims each")
print()
g = group_db[0]
print(f"Sample — Group {g['id']}")
print(f"  commodities  : {g['meta']['commodities']}")
print(f"  target_roles : {g['meta']['target_roles']}")
print(f"  lat/lon      : {g['meta']['latitude']}, {g['meta']['longitude']}")
print(f"  vector       : {g['vector'].round(4)}")
print(f"    [{COMM_DIMS}]")
print(f"    [{ROLE_DIMS}]")
print(f"    [{GEO_DIMS}]")


Group vector DB : 200 records,  9 dims each

Sample — Group 1
  commodities  : rice
  target_roles : trader
  lat/lon      : 21.1458, 79.0882
  vector       : [0.     1.     0.     0.     0.     1.5    0.5297 2.7474 1.0822]
    [['has_cotton', 'has_rice', 'has_sugar']]
    [['targets_broker', 'targets_exporter', 'targets_trader']]
    [['geo_x', 'geo_y', 'geo_z']]


## Search function

In [8]:
def search_group(group_id, top_k=15):
    """
    Find the top_k most similar groups to the given group.

    Similarity is cosine similarity over the 9-dim vector:
      commodity section  — rewards shared commodities        (COMMODITY_BOOST={cb})
      role section       — rewards same target roles         (ROLE_BOOST={rb})
      geo section        — rewards geographic proximity      (GEO_BOOST={gb})

    The query group is excluded from results.
    """.format(cb=COMMODITY_BOOST, rb=ROLE_BOOST, gb=GEO_BOOST)

    # 1. Locate query group
    query = next((g for g in group_db if g['id'] == group_id), None)
    if query is None:
        print(f"Group {group_id} not found.")
        return pd.DataFrame()

    q_vec = query['vector'].reshape(1, -1)

    # 2. All other groups as candidates
    candidates = [g for g in group_db if g['id'] != group_id]
    cand_vecs  = np.array([g['vector'] for g in candidates])

    # 3. Cosine similarity
    sims = cosine_similarity(q_vec, cand_vecs)[0]

    # 4. Sort and build results table
    top_idx = np.argsort(sims)[::-1][:top_k]
    rows = []
    for i in top_idx:
        m = candidates[i]['meta']
        rows.append({
            'group_id':    m['group_id'],
            'commodities': m['commodities'],
            'target_roles': m['target_roles'],
            'latitude':    round(m['latitude'], 4),
            'longitude':   round(m['longitude'], 4),
            'similarity':  round(float(sims[i]), 4),
        })

    return pd.DataFrame(rows)


## Test run

In [10]:
TEST_GROUP_ID = 1

q = next(g['meta'] for g in group_db if g['id'] == TEST_GROUP_ID)
print(f"Query  — Group {q['group_id']} | {q['commodities']} | targets: {q['target_roles']} "
      f"| {q['latitude']:.4f}, {q['longitude']:.4f}")
print("=" * 70)

results = search_group(TEST_GROUP_ID, top_k=15)
print(results.to_string(index=False))

print("\nWhat to verify:")
print("1. Top results share the same commodities as the query group")
print("2. Top results share the same target roles as the query group")
print("3. Among equally matching groups, geographically closer ones rank higher")


Query  — Group 1 | rice | targets: trader | 21.1458, 79.0882
 group_id commodities target_roles  latitude  longitude  similarity
       27        rice       trader   21.1458    79.0882         1.0
       28        rice       trader   21.1458    79.0882         1.0
       29        rice       trader   21.1458    79.0882         1.0
       30        rice       trader   21.1458    79.0882         1.0
       31        rice       trader   21.1458    79.0882         1.0
       32        rice       trader   21.1458    79.0882         1.0
       33        rice       trader   21.1458    79.0882         1.0
       17        rice       trader   21.1458    79.0882         1.0
       16        rice       trader   21.1458    79.0882         1.0
       15        rice       trader   21.1458    79.0882         1.0
       14        rice       trader   21.1458    79.0882         1.0
       13        rice       trader   21.1458    79.0882         1.0
       12        rice       trader   21.1458    79.0882

## Visualisation

In [11]:
SEC_COLORS = {
    'commodity': '#7F77DD',
    'role':      '#1D9E75',
    'geo':       '#EF9F27',
}
SEC_SLICES = {
    'commodity': slice(0,       n_comm),
    'role':      slice(n_comm,  n_comm + n_role),
    'geo':       slice(n_comm + n_role, None),
}


def visualize_group_search(group_id, top_k=8):
    # ── 1. Data ──────────────────────────────────────────────────────────────
    query   = next(g for g in group_db if g['id'] == group_id)
    q_meta  = query['meta']
    q_vec   = query['vector']

    results  = search_group(group_id, top_k=top_k)
    top_ids  = results['group_id'].tolist()
    sims     = results['similarity'].tolist()

    cand_records = sorted(
        [g for g in group_db if g['id'] in top_ids],
        key=lambda g: top_ids.index(g['id'])
    )
    cand_vecs  = np.array([g['vector'] for g in cand_records])
    cand_short = [
        f"G{g['id']} {g['meta']['commodities'][:8]}"
        for g in cand_records
    ]

    # ── 2. Layout ─────────────────────────────────────────────────────────────
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            '① Radar — query vs top groups',
            '② Heatmap — all 9 dims',
            '③ PCA scatter — all 200 groups',
            '④ Score breakdown by section',
        ],
        specs=[[{'type': 'polar'}, {'type': 'xy'}],
               [{'type': 'xy'},    {'type': 'xy'}]],
        vertical_spacing=0.14,
        horizontal_spacing=0.10,
    )

    # ── ① RADAR ───────────────────────────────────────────────────────────────
    all_vecs = np.vstack([q_vec, cand_vecs])
    vmax     = all_vecs.max(axis=0);  vmax[vmax == 0] = 1
    q_norm   = q_vec    / vmax
    c_norms  = cand_vecs / vmax
    theta    = DIM_LABELS + [DIM_LABELS[0]]

    fig.add_trace(go.Scatterpolar(
        r=np.append(q_norm, q_norm[0]), theta=theta,
        name=f'Group {group_id} (query)',
        line=dict(color='#534AB7', width=2.5),
        fill='toself', fillcolor='rgba(83,74,183,0.12)',
    ), row=1, col=1)

    for i, (cv, lbl, sim) in enumerate(zip(c_norms, cand_short, sims)):
        is_top = (i == 0)
        fig.add_trace(go.Scatterpolar(
            r=np.append(cv, cv[0]), theta=theta,
            name=f'{lbl}  {sim:.3f}',
            line=dict(
                color='rgba(15,110,86,0.85)' if is_top else 'rgba(136,135,128,0.35)',
                width=1.8 if is_top else 1,
            ),
            fill='toself' if is_top else None,
            fillcolor='rgba(15,110,86,0.08)' if is_top else None,
        ), row=1, col=1)

    # ── ② HEATMAP ─────────────────────────────────────────────────────────────
    heat_matrix = np.vstack([q_vec, cand_vecs])
    heat_labels = [f'★ Group {group_id}'] + cand_short

    fig.add_trace(go.Heatmap(
        z=heat_matrix, x=DIM_LABELS, y=heat_labels,
        colorscale='Purples', showscale=True,
        xgap=1, ygap=1,
        hovertemplate='%{y}<br>%{x}: %{z:.3f}<extra></extra>',
    ), row=1, col=2)

    # ── ③ PCA SCATTER ─────────────────────────────────────────────────────────
    all_stored = np.array([g['vector'] for g in group_db])
    all_gids   = np.array([g['id']     for g in group_db])

    pca    = PCA(n_components=2, random_state=42)
    proj   = pca.fit_transform(all_stored)
    q_proj = pca.transform(q_vec.reshape(1, -1))[0]

    top_mask  = np.isin(all_gids, top_ids)
    rest_mask = ~top_mask

    fig.add_trace(go.Scatter(
        x=proj[rest_mask, 0], y=proj[rest_mask, 1],
        mode='markers',
        marker=dict(color='rgba(136,135,128,0.25)', size=6),
        name='Other groups', showlegend=False,
        hovertemplate='Group %{text}<extra></extra>',
        text=all_gids[rest_mask].astype(str),
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=proj[top_mask, 0], y=proj[top_mask, 1],
        mode='markers+text',
        marker=dict(color='#1D9E75', size=10),
        text=all_gids[top_mask].astype(str),
        textposition='top center', textfont=dict(size=9),
        name='Top matches',
        hovertemplate='Group %{text}<extra></extra>',
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=[q_proj[0]], y=[q_proj[1]],
        mode='markers+text',
        marker=dict(color='#534AB7', size=16, symbol='star'),
        text=[f'G{group_id}'], textposition='top center',
        name='Query group',
    ), row=2, col=1)

    var = pca.explained_variance_ratio_
    fig.add_annotation(
        x=0.27, y=0.03, xref='paper', yref='paper',
        text=f'PC1 {var[0]*100:.1f}%  PC2 {var[1]*100:.1f}% variance',
        showarrow=False, font=dict(size=10, color='gray'),
    )

    # ── ④ SCORE BREAKDOWN ─────────────────────────────────────────────────────
    q_norm_f = np.linalg.norm(q_vec) + 1e-9
    for sec, sl in SEC_SLICES.items():
        contribs = [
            float(np.dot(q_vec[sl], cv[sl]) /
                  (q_norm_f * np.linalg.norm(cv) + 1e-9))
            for cv in cand_vecs
        ]
        fig.add_trace(go.Bar(
            x=cand_short, y=contribs,
            name=sec, marker_color=SEC_COLORS[sec],
            hovertemplate=f'{sec}: %{{y:.4f}}<extra></extra>',
        ), row=2, col=2)

    # ── Layout ────────────────────────────────────────────────────────────────
    fig.update_layout(
        height=840,
        title=dict(
            text=(f'<b>Group {group_id}</b>  |  '
                  f'commodities: {q_meta["commodities"]}  |  '
                  f'targets: {q_meta["target_roles"]}  |  '
                  f'lat {q_meta["latitude"]:.4f}, lon {q_meta["longitude"]:.4f}'),
            font=dict(size=13),
        ),
        barmode='stack',
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        legend=dict(font=dict(size=10), x=1.01, y=0.99),
        margin=dict(r=180),
    )
    fig.update_xaxes(tickangle=-40, tickfont=dict(size=9), row=1, col=2)
    fig.update_yaxes(tickfont=dict(size=9), row=1, col=2)
    fig.update_xaxes(tickangle=-35, tickfont=dict(size=9), row=2, col=2)

    fig.show(renderer="browser")


## Run

In [12]:
# Change TEST_GROUP_ID to any group_id from 1–200
TEST_GROUP_ID = 1

q = next(g['meta'] for g in group_db if g['id'] == TEST_GROUP_ID)
print(f"Query — Group {q['group_id']} | {q['commodities']} "
      f"| targets: {q['target_roles']}")
print("=" * 60)
print(search_group(TEST_GROUP_ID, top_k=10).to_string(index=False))


Query — Group 1 | rice | targets: trader
 group_id commodities target_roles  latitude  longitude  similarity
       27        rice       trader   21.1458    79.0882         1.0
       28        rice       trader   21.1458    79.0882         1.0
       29        rice       trader   21.1458    79.0882         1.0
       30        rice       trader   21.1458    79.0882         1.0
       31        rice       trader   21.1458    79.0882         1.0
       32        rice       trader   21.1458    79.0882         1.0
       33        rice       trader   21.1458    79.0882         1.0
       17        rice       trader   21.1458    79.0882         1.0
       16        rice       trader   21.1458    79.0882         1.0
       15        rice       trader   21.1458    79.0882         1.0


In [14]:
visualize_group_search(TEST_GROUP_ID, top_k=8 , )


In [15]:
# Try a group with multi-commodity + multi-role to see richer matches
visualize_group_search(101, top_k=8)
